# 🧬 Clinical Trial RL Agent — GRPO Training
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Roopalgn/openenv-clinical-trial/blob/main/train_colab.ipynb)

**Environment**: [HF Space](https://huggingface.co/spaces/Roopalgn/openenv-clinical-trial) · **Model**: Qwen2.5-1.5B · **Method**: GRPO via Unsloth + TRL

An RL agent learns to design statistically rigorous clinical trials — selecting endpoints, enrolling patients, running Phase I safety, interim and primary analyses — in the correct order and with adequate statistical power.

**Key design**: Full-episode evaluation. The model outputs a complete 10-action plan which is executed against the live environment, giving GRPO a 19-point reward range (−3 → +16) vs the degenerate 2.75-point range of single-step evaluation.

In [ ]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

!pip install unsloth trl datasets requests -q 2>/dev/null
print('✓ Dependencies ready')

## 1 · Connect to Live Environment

The environment runs as an HTTP API on Hugging Face Spaces. No local server setup needed.

In [ ]:
import requests, json, re, math

ENV_URL = 'https://roopalgn-openenv-clinical-trial.hf.space'

ping = requests.get(f'{ENV_URL}/ping', timeout=15).json()
print(f'Environment status: {ping}')

obs = requests.post(f'{ENV_URL}/reset', json={'seed': 42}, timeout=30).json()
print(f"Phase  : {obs.get('phase_data', {}).get('current_phase')}")
print(f"Actions: {obs.get('available_actions')}")
print(f"Budget : {obs.get('resource_status', {}).get('budget_remaining')}")

## 2 · Helper Functions

- `parse_action_plan`: extracts a list of actions from model JSON output (handles fenced blocks, malformed JSON)
- `full_episode_reward`: executes the complete plan against the environment and sums cumulative reward

In [ ]:
MAX_EPISODE_STEPS = 20

def env_reset(seed=None):
    r = requests.post(f'{ENV_URL}/reset', json={'seed': seed} if seed else {}, timeout=30)
    r.raise_for_status(); return r.json()

def env_step(action_type, parameters=None, justification='', confidence=0.7):
    r = requests.post(f'{ENV_URL}/step',
        json={'action_type': action_type, 'parameters': parameters or {},
              'justification': justification, 'confidence': confidence}, timeout=30)
    r.raise_for_status(); return r.json()

def _normalize(a):
    at = str(a.get('action_type', '')).strip()
    params = a.get('parameters', {})
    if not isinstance(params, dict): params = {}
    if at == 'enroll_patients' and 'n_patients' not in params:
        params['n_patients'] = params.pop('sample_size', 240)
    return {'action_type': at, 'parameters': params,
            'justification': str(a.get('justification', '')),
            'confidence': float(a.get('confidence', 0.7))}

def parse_action_plan(text):
    candidates = re.findall(r'```(?:json)?\s*(.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
    candidates.append(text)
    for c in candidates:
        s, e = c.find('{'), c.rfind('}')
        if s == -1 or e <= s: continue
        try:
            parsed = json.loads(c[s:e+1])
        except Exception:
            continue
        if 'actions' in parsed and isinstance(parsed['actions'], list):
            acts = [_normalize(a) for a in parsed['actions'] if isinstance(a, dict) and 'action_type' in a]
            if acts: return acts
        if 'action_type' in parsed: return [_normalize(parsed)]
    return []

def extract_reward(result):
    r = result.get('reward', 0.0)
    if isinstance(r, dict): return float(sum(float(v) for v in r.values()))
    return float(r)

def full_episode_reward(text, seed):
    try:
        actions = parse_action_plan(text)
        if not actions: return -3.0
        env_reset(seed=seed)
        total = 0.0
        for action in actions[:MAX_EPISODE_STEPS]:
            try:
                result = env_step(action['action_type'], action.get('parameters', {}),
                                  action.get('justification', ''), action.get('confidence', 0.7))
                total += extract_reward(result)
                if result.get('observation', result).get('done', False): break
            except Exception: total -= 0.5
        return total
    except Exception: return -3.0

print('✓ Helper functions ready')

## 3 · Dry Run — Reward Discrimination Check

Before training, we verify the reward signal is discriminative. GRPO needs large spread between good and bad completions.

Expected: **good plan ≈ +15**, **minimal plan ≈ +0.5**, **parse failure = −3**

In [ ]:
GOOD_PLAN = json.dumps({'actions': [
    {'action_type': 'set_primary_endpoint', 'parameters': {'endpoint': 'overall_survival'}, 'justification': 'OS gold standard', 'confidence': 0.9},
    {'action_type': 'set_sample_size', 'parameters': {'sample_size': 240}, 'justification': '80% power', 'confidence': 0.85},
    {'action_type': 'set_inclusion_criteria', 'parameters': {'criteria': 'adults 18-75'}, 'justification': 'standard', 'confidence': 0.8},
    {'action_type': 'set_dosing_schedule', 'parameters': {'schedule': 'daily'}, 'justification': 'standard', 'confidence': 0.8},
    {'action_type': 'set_control_arm', 'parameters': {'control': 'placebo'}, 'justification': 'RCT', 'confidence': 0.9},
    {'action_type': 'enroll_patients', 'parameters': {'n_patients': 240}, 'justification': 'full enrollment', 'confidence': 0.85},
    {'action_type': 'run_dose_escalation', 'parameters': {}, 'justification': 'Phase I safety', 'confidence': 0.8},
    {'action_type': 'run_interim_analysis', 'parameters': {}, 'justification': 'futility check', 'confidence': 0.75},
    {'action_type': 'run_primary_analysis', 'parameters': {}, 'justification': 'final test', 'confidence': 0.8},
    {'action_type': 'synthesize_conclusion', 'parameters': {}, 'justification': 'complete', 'confidence': 0.85},
]})

print('Running discrimination check...')
deltas = []
for ep in range(3):
    seed = 42 + ep
    r_good = full_episode_reward(GOOD_PLAN, seed)
    r_minimal = full_episode_reward(json.dumps({'actions': [
        {'action_type': 'set_primary_endpoint', 'parameters': {}, 'justification': 'test', 'confidence': 0.5}]}), seed)
    r_fail = full_episode_reward('not json', seed)
    delta = r_good - r_minimal
    deltas.append(delta)
    print(f'  Ep {ep+1}: good={r_good:+.2f}  minimal={r_minimal:+.2f}  fail={r_fail:+.2f}  delta={delta:+.2f}')

avg = sum(deltas)/len(deltas)
status = '✓ READY' if avg > 2.0 else '✗ LOW'
print(f'\nAvg delta: {avg:.2f}  {status} for training')

## 4 · Load Model and Configure GRPO

Using Qwen2.5-1.5B-Instruct in 4-bit with LoRA (r=8). GRPO evaluates 6 completions per step and updates the policy based on relative rewards.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit',
    max_seq_length=2048, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=8, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', use_gradient_checkpointing='unsloth',
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'✓ Model loaded  |  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are an expert clinical trial designer.
Plan the COMPLETE sequence of actions to design a successful trial.
Return a JSON object with an \"actions\" list (8-12 actions).

Example:
{\"actions\": [
  {\"action_type\": \"set_primary_endpoint\", \"parameters\": {\"endpoint\": \"overall_survival\"}, \"justification\": \"OS gold standard\", \"confidence\": 0.9},
  {\"action_type\": \"set_sample_size\", \"parameters\": {\"sample_size\": 240}, \"justification\": \"80% power\", \"confidence\": 0.85},
  {\"action_type\": \"set_inclusion_criteria\", \"parameters\": {\"criteria\": \"adults 18-75\"}, \"justification\": \"standard\", \"confidence\": 0.8},
  {\"action_type\": \"set_dosing_schedule\", \"parameters\": {\"schedule\": \"daily\"}, \"justification\": \"standard\", \"confidence\": 0.8},
  {\"action_type\": \"set_control_arm\", \"parameters\": {\"control\": \"placebo\"}, \"justification\": \"RCT\", \"confidence\": 0.9},
  {\"action_type\": \"enroll_patients\", \"parameters\": {\"n_patients\": 240}, \"justification\": \"full enrollment\", \"confidence\": 0.85},
  {\"action_type\": \"run_dose_escalation\", \"parameters\": {}, \"justification\": \"Phase I safety\", \"confidence\": 0.8},
  {\"action_type\": \"run_interim_analysis\", \"parameters\": {}, \"justification\": \"futility check\", \"confidence\": 0.75},
  {\"action_type\": \"run_primary_analysis\", \"parameters\": {}, \"justification\": \"final test\", \"confidence\": 0.8},
  {\"action_type\": \"synthesize_conclusion\", \"parameters\": {}, \"justification\": \"complete trial\", \"confidence\": 0.85}
]}

Include ALL phases: design -> enrollment -> analysis -> conclusion.
Return ONLY the JSON. No extra text."""

N_PROMPTS = 20
chat_prompts = []
for i in range(N_PROMPTS):
    obs = env_reset(seed=42 + i)
    phase = obs.get('phase_data', {}).get('current_phase', 'design')
    resources = obs.get('resource_status', {})
    scenario = obs.get('scenario_description', '')
    user_msg = (f'Scenario: {scenario}\nPhase: {phase}\nResources: {json.dumps(resources)}\n\n'
                f'Plan the complete trial (8-12 actions). Return ONLY JSON.')
    chat_prompts.append({'prompt': [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_msg},
    ]})

train_dataset = Dataset.from_list(chat_prompts)
print(f'✓ Dataset ready: {len(train_dataset)} prompts from {N_PROMPTS} unique seeds')

## 5 · Train with GRPO

The reward function executes each completion as a full episode (up to 20 env steps) and returns the cumulative reward. GRPO updates the policy to favour completions that score higher within each group of 6.

In [ ]:
import torch
from trl import GRPOConfig, GRPOTrainer

seed_counter = [0]

def reward_func(completions, **kwargs):
    rewards = []
    for c in completions:
        text = c[-1]['content'] if isinstance(c, list) else str(c)
        seed = 99999 + seed_counter[0]; seed_counter[0] += 1
        rewards.append(full_episode_reward(text, seed))
    return rewards

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = GRPOConfig(
    output_dir='checkpoints/grpo',
    num_generations=6,
    max_completion_length=1024,
    temperature=0.5,
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=6,
    max_steps=30,
    warmup_steps=2,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=1,
    save_steps=10,
    report_to='none',
    bf16=use_bf16,
    fp16=not use_bf16,
)

trainer = GRPOTrainer(
    model=model, args=training_args, tokenizer=tokenizer,
    train_dataset=train_dataset, reward_funcs=[reward_func],
)

print('Starting GRPO training (30 steps, 6 generations/step, ~75 min on T4)...')
trainer.train()
print('\n✓ Training complete!')

## 6 · Results

The cell below plots the reward curve. If you ran training above, it uses your live results. Otherwise it displays our recorded training run.

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import numpy as np, csv, os

# ── Load rewards ──────────────────────────────────────────────────────────────
csv_path = 'checkpoints/grpo/reward_log.csv'
if os.path.exists(csv_path):
    rows = list(csv.DictReader(open(csv_path)))
    steps   = [int(r['step']) for r in rows]
    rewards = [float(r['reward']) for r in rows]
    print('Loaded rewards from live training run.')
else:
    # Actual results from our recorded training run
    rewards = [12.05, 4.705, 3.678, 7.759, 6.167, 7.558, 3.049, 6.781, 11.7, 9.159,
               12.2, -0.763, 7.292, 9.08, 2.177, 12.29, 12.18, 3.384, 7.778, 8.041,
               -0.430, 11.77, 11.42, 12.78, 11.31, 3.066, 1.858, 9.097, 12.19, 8.072]
    steps   = list(range(1, 31))
    print('Displaying recorded training run results.')

# ── Compute stats ─────────────────────────────────────────────────────────────
z      = np.polyfit(steps, rewards, 1)
trend  = np.poly1d(z)
window = [np.mean(rewards[max(0,i-4):i+1]) for i in range(len(rewards))]
r1, r2, r3 = np.mean(rewards[:10]), np.mean(rewards[10:20]), np.mean(rewards[20:])

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.fill_between(steps, rewards, alpha=0.08, color='#4C72B0')
ax.plot(steps, rewards, 'o', alpha=0.55, color='#4C72B0', ms=5, label='Step reward')
ax.plot(steps, trend(steps), '--', color='#DD4444', lw=2.2, label=f'Trend  {z[0]:+.3f}/step')
ax.plot(steps, window, '-', color='#2CA02C', lw=2, label='5-step rolling avg')
ax.axhline(0, color='gray', ls=':', alpha=0.4)
for x, y, label in [(5, r1, f'{r1:.1f}'), (15, r2, f'{r2:.1f}'), (25, r3, f'{r3:.1f}')]:
    ax.annotate(f'avg {label}', xy=(x, y+0.7), ha='center', fontsize=8.5,
                color='#7B2D8B', fontweight='bold')
ax.set_xlabel('Training Step'); ax.set_ylabel('Episode Reward')
ax.set_title(
    f'GRPO Training  ·  Qwen2.5-1.5B  ·  Clinical Trial Agent\n'
    f'Mean reward: {np.mean(rewards):+.2f}  |  Slope: {z[0]:+.4f}/step  |  '
    f'Collapsed steps: 0 / {len(steps)}  |  Rolling avg: {r1:.1f} → {r2:.1f} → {r3:.1f}',
    fontsize=10)
ax.legend(fontsize=9); ax.grid(alpha=0.25); plt.tight_layout()
os.makedirs('docs', exist_ok=True)
plt.savefig('docs/reward_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → docs/reward_plot.png')

## 7 · Summary

| Metric | Value |
|---|---|
| Mean episode reward | **+7.58** |
| Trend slope | **+0.055 / step** ✅ |
| Collapsed steps (`reward_std = 0`) | **0 / 30** ✅ |
| Rolling avg: steps 1–10 | +7.26 |
| Rolling avg: steps 11–20 | +7.37 |
| Rolling avg: steps 21–30 | **+8.11** (highest) |
| Peak step reward | **+12.78** (step 24) |

### What fixed the flat reward curve

| Problem | Fix |
|---|---|
| Single-step eval → 2.75pt range → zero GRPO gradient | **Full-episode eval** → 19pt range |
| 512-token limit → JSON truncated → parse fail → all −3 | **1024 tokens** → completions finish naturally |
| Weak milestone bonuses → noise > signal | **Doubled milestones** + progress bonus |
| Last-step clean after 10 violations | **Episode-wide violation penalty** at terminal |